# Base install smoke check

Run this notebook in the Python environment where SnowIn is installed. It checks which SnowIn installation the kernel imports, confirms the declared base requirements, and runs a small synthetic dSWE calculation against the Leinss equation. A subprocess repeats the core calculation with optional NISAR, Dask, and geospatial imports simulated as unavailable.

No product files, network access, plotting, or DEM are needed. This checks the core install path; it does not run the optional GUNW reader or local-incidence workflow.

In [1]:
import importlib.metadata
import re
import sys

import numpy as np
import xarray as xr
import snowin
from snowin import compute_dswe

distribution = importlib.metadata.distribution("snowin")
requirements = distribution.requires or []
base_requirements = set()
optional_requirements = []
for requirement in requirements:
    if "extra ==" in requirement:
        optional_requirements.append(requirement)
        continue
    name = re.match(r"\s*([A-Za-z0-9_.-]+)", requirement).group(1)
    base_requirements.add(re.sub(r"[-_.]+", "-", name.lower()))

assert base_requirements == {"numpy", "xarray"}, base_requirements
print(f"SnowIn version: {snowin.__version__}")
print(f"Imported module: {snowin.__name__}")
print(f"Base requirements: {sorted(base_requirements)}")
print(f"Optional dependency declarations: {len(optional_requirements)}")

SnowIn version: 0.1.0
Imported module: snowin
Base requirements: ['numpy', 'xarray']
Optional dependency declarations: 33


In [2]:
phase_value = 0.75
incidence_rad = np.deg2rad(35.0)
wavelength_m = 0.238403545

phase = xr.DataArray(
    [[phase_value]],
    dims=("y", "x"),
    coords={"y": [0.0], "x": [0.0]},
    attrs={
        "units": "rad",
        "phase_difference_definition": "secondary_minus_reference",
    },
)
incidence = xr.DataArray(
    [[incidence_rad]],
    dims=("y", "x"),
    coords=phase.coords,
    attrs={"units": "rad", "incidence_angle_reference": "local"},
)

result = compute_dswe(phase, incidence, wavelength_m=wavelength_m)
expected_m = phase_value * wavelength_m / (
    2 * np.pi * (1.59 + incidence_rad**2.5)
)
np.testing.assert_allclose(result.item(), expected_m, rtol=1e-12)
assert result.attrs["units"] == "m"
print(f"dSWE: {result.item():.12g} {result.attrs['units']}")
print(f"dSWE: {result.item() * 1000:.9f} mm")

dSWE: 0.0151235943162 m
dSWE: 15.123594316 mm


## Core calculation with optional imports unavailable

The next cell starts a clean Python subprocess and makes optional modules behave as if they are not installed. NumPy and xarray remain available as the declared base dependencies. The parent notebook kernel itself may have the optional packages installed.

In [3]:
import subprocess
import textwrap

optional_probe = textwrap.dedent("""\
    import sys
    for name in (
        "dask", "fsspec", "geopandas", "h5netcdf", "h5py",
        "matplotlib", "nisar_pytools", "pyproj", "rasterio",
        "rioxarray", "s3fs", "scipy", "shapely",
    ):
        sys.modules[name] = None
    import numpy as np
    import xarray as xr
    import snowin
    from snowin import compute_dswe
    phase = xr.DataArray(
        [[0.75]], dims=("y", "x"),
        attrs={"units": "rad", "phase_difference_definition": "secondary_minus_reference"},
    )
    incidence = xr.DataArray(
        [[np.deg2rad(35.0)]], dims=("y", "x"),
        attrs={"units": "rad", "incidence_angle_reference": "local"},
    )
    result = compute_dswe(phase, incidence, wavelength_m=0.238403545)
    assert np.isfinite(result.values).all()
    assert result.attrs["units"] == "m"
    assert all(sys.modules.get(name) is None for name in ("dask", "h5py", "nisar_pytools", "rasterio", "scipy"))
    print(f"core dSWE smoke passed: {result.item():.12g} {result.attrs['units']}")
    print(f"SnowIn module: {snowin.__name__}")
""")
completed = subprocess.run(
    [sys.executable, "-c", optional_probe],
    capture_output=True,
    text=True,
)
print(completed.stdout, end="")
if completed.returncode != 0:
    raise RuntimeError(completed.stderr or "optional-dependency smoke failed")

core dSWE smoke passed: 0.0151235943162 m
SnowIn module: snowin


## How to read the result

A passing notebook confirms that the active SnowIn installation can run the core xarray dSWE path using NumPy and xarray, and that importing the core package does not require the optional adapters. The synthetic value is **0.015123594316 m**, or **15.123594 mm**. It is a smoke-test input, not a real-product retrieval or scientific validation.